For informative purposes. Only Reaxys users can download the input data used in this notebook. Details on which data was used are availabl in the Methods section of the corresponding publication (see README).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# First standardisation of species names

Done at `/aloy/home/grojas/tesis/antibiotics/final_pipeline/github_repo/antibiotics_design/src/reaxys_data_preprocessing/process_species_names_with_reference_list.py`

# Data filtering and binarization

In [ ]:
# private_file = raw Reaxys data with standardised names
data = pd.read_csv(<private_file>)
data.head()

In [ ]:
data.shape

(2408573, 39)

In [ ]:
data.fillna("", inplace=True)

/tmp/ipykernel_270720/370793892.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  data.fillna("", inplace=True)


## Filter AFTYPES

In [ ]:
aftypes_interest = list(data.AFTYPE.str.lower().value_counts().head(15).index)
aftypes_interest

In [ ]:
aftypes_interest = [
    "bacterial growth/proliferation/viability",
    "cytotoxicity",
    "in vitro",
    "",
    "fungal growth/proliferation/viability",
    "antibacterial activity",
    "antimicrobial activity",
    "antifungal activity",
    "antimycobacterial activity",
    "antitubercular activity",
    "cell/tumor cell: proliferation/viability/growth",
    "bactericidal activity",
]

In [ ]:
data["AFTYPE"] = data["AFTYPE"].str.lower()

In [ ]:
data.shape

(2408573, 39)

In [ ]:
data = data[data["AFTYPE"].isin(aftypes_interest)]
data.shape

(2350660, 39)

## Filter VTYPES

In [ ]:
data["VTYPE"] = data["VTYPE"].str.lower()

In [ ]:
vtypes_interest = list(data.VTYPE.value_counts().head(19).index)
vtypes_interest

['mic',
 'zi (zone of inhibition)',
 'qualitative',
 'inhibition percentage',
 'mic90',
 'mbc (minimum bactericidal concentration)',
 'mic50',
 'inhibition rate',
 'ic50',
 'mic80',
 'mfc (minimum fungicidal concentration)',
 'tested',
 'ec50',
 'mic100',
 'ic90',
 'mic95',
 'pic50',
 'percentage',
 'activity']

In [ ]:
data = data[
    data["VTYPE"].isin(
        [
            'mic',
    'zi (zone of inhibition)',
    'qualitative',
    'inhibition percentage',
    'mic90',
    'mbc (minimum bactericidal concentration)',
    'mic50',
    'inhibition rate',
    'ic50',
    'mic80',
    'mfc (minimum fungicidal concentration)',
    'ec50',
    'mic100',
    'ic90',
    'mic95',
    'pic50',
    'percentage',
    'activity',
        ]
    )
]

## Explore units and compute PAUREUS if possible for missing values

In [ ]:
import numpy as np

from rdkit import Chem
from rdkit.Chem import Descriptors


def get_molecular_weight(smiles):
    """Calcula el peso molecular (MW) a partir de un SMILES"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("SMILES inválido.")
    return Descriptors.MolWt(mol)


def convert_to_molarity(value, unit, smiles, mw=None):
    """
    Converts to molarity (M) a concentration given in different units.
    """
    unit_ = unit.lower()

    if unit_ in ["ug/ml", "μg/ml"]:
        if mw is None:
            mw = get_molecular_weight(smiles)
        return (value * 1e-6) / mw  # Convierte a gramos por litro y divide por MW

    elif unit_ == "ng/ml":
        if mw is None:
            mw = get_molecular_weight(smiles)
        return (value * 1e-9) / mw

    elif unit in ["uM", "µM"]:
        return value * 1e-6  # µM a M

    elif unit_ in ["ug/mg/ml", "μg/mg/ml"]:
        if mw is None:
            mw = get_molecular_weight(smiles)
        return (
            value * 1e-6
        ) / mw  # Similar a ug/mL si la densidad es asumida como 1 mg/mL

    elif unit_ in ["ug/kg", "μg/kg"]:
        if mw is None:
            mw = get_molecular_weight(smiles)
        return (value * 1e-6) / mw  # Suponiendo que es masa en 1 kg de solución

    elif unit_ == "mg/l":
        if mw is None:
            mw = get_molecular_weight(smiles)
        return (value * 1e-3) / mw  # mg a gramos, luego dividir por MW

    elif unit == "mM":  # original unit, lowercase can also mean milimeters
        return value * 1e-3  # mM a M

    elif unit_ == "ppm":
        if mw is None:
            mw = get_molecular_weight(smiles)
        return (value * 1e-6) / mw  # ppm = mg/L, equivalente a ug/mL

    elif unit_ == "mg/ml":
        if mw is None:
            mw = get_molecular_weight(smiles)
        return (value * 1e-3) / mw  # mg a gramos, luego dividir por MW

    else:
        raise ValueError(f"Unvalid unit: {unit}")


def convert_to_paureus(value, unit, mw=None):
    """
    Converts from molearity to PAUREUS.
    """
    molarity = convert_to_molarity(value, unit, mw)
    return -np.log10(molarity)

In [ ]:
from tqdm import tqdm
new_paureus = []
for i in tqdm(range(data.shape[0])):
    paureus = data.PAUREUS.values[i]
    if paureus != "":
        new_paureus.append(paureus)
    else:
        try:
            value = float(data.VALUE.values[i])
        except:
            new_paureus.append("")
            continue
        unit = data.UNIT.values[i]
        smiles = data.smiles.values[i]
        try:
            new_paureus.append(convert_to_paureus(value, unit, smiles))
        except:
            new_paureus.append("")

  1%|          | 12793/2194772 [00:00<02:36, 13964.29it/s]/tmp/ipykernel_270720/1810208484.py:73: RuntimeWarning: divide by zero encountered in log10
  return -np.log10(molarity)
 11%|█         | 239792/2194772 [00:05<00:47, 40745.39it/s]/tmp/ipykernel_270720/1810208484.py:73: RuntimeWarning: invalid value encountered in log10
  return -np.log10(molarity)
 21%|██        | 463185/2194772 [00:10<00:30, 56451.33it/s][14:24:03] WARNING: not removing hydrogen atom without neighbors
[14:24:03] WARNING: not removing hydrogen atom without neighbors
[14:24:03] WARNING: not removing hydrogen atom without neighbors
[14:24:03] WARNING: not removing hydrogen atom without neighbors
 87%|████████▋ | 1917845/2194772 [00:47<00:13, 20182.89it/s][14:24:39] Explicit valence for atom # 24 P, 6, is greater than permitted
[14:24:39] Explicit valence for atom # 24 P, 6, is greater than permitted
[14:24:39] Explicit valence for atom # 24 P, 6, is greater than permitted
[14:24:39] Explicit valence for atom # 24

In [ ]:
sum(data["PAUREUS"] == "") - sum(
    np.array(new_paureus) == ""
)  # new PAUREUS values added

np.int64(49074)

In [ ]:
data["new PAUREUS"] = new_paureus

## Binarize

In [ ]:
data.reset_index(drop=True, inplace=True)

In [ ]:
# auxiliary matrix
df = data[["PAUORIG", "EXACT", "TEXT", "PAUREUS", "VTYPE", "VALUE", "UNIT"]].copy()


# Binarize based on 'PAUORIG' column
def binarize_row(row):
    active_words = [
        "active",
        "highly active",
        "moderately active",
        "strongly active",
        "more active",
        "most active",
        "highly active",
        "highly sensitive",
        "partially active",
        "very active",
        "modest activity",
    ]
    inactive_words = [
        "inactive",
        "no activity",
        "resistant",
        "weakly active",
        "slightly active",
        "not active",
        "least active",
        "NA",
        "not observed",
        "not sensitive",
        "fairly sensitive",
        "feebly sensitive",
        "low activity",
        "less active",
        "in active",
        "no active",
    ]

    if row["PAUORIG"] == "":
        row["PAUORIG"] = None
    elif (
        "Non Active" in row["PAUORIG"]
        or "pAur < 5" in row["PAUORIG"]
        or "%inhib < 25" in row["PAUORIG"]
    ):
        row["PAUORIG"] = 0
    else:
        row["PAUORIG"] = None

    if row["EXACT"] == "":
        row["EXACT"] = None
    elif (
        "Low" in row["EXACT"]
        or "Not determined/not detected" in row["EXACT"]
        or "Not active" in row["EXACT"]
    ):
        row["EXACT"] = 0
    elif (
        "Active" in row["EXACT"] or "High" in row["EXACT"] or "Moderate" in row["EXACT"]
    ):
        row["EXACT"] = 1
    else:
        row["EXACT"] = None

    if row["TEXT"] == "":
        row["TEXT"] = None
    elif any(word in row["TEXT"].lower() for word in inactive_words):
        row["TEXT"] = 0
    elif any(word in row["TEXT"].lower() for word in active_words):
        row["TEXT"] = 1

    else:
        row["TEXT"] = None

    # Binarize data coming from % inhibition (80% threshold)
    if (
        row["VTYPE"] in ["inhibition percentage", "inhibition rate"]
        and row["UNIT"] == "%"
    ):
        try:
            v = float(row["VALUE"])
            if v > 80:
                row["VALUE"] = 1
            elif v < 80:
                row["VALUE"] = 0
            else:
                row["VALUE"] = None
        except:
            row["VALUE"] = None
    else:
        row["VALUE"] = None

    if row["PAUREUS"] == "":
        row["PAUREUS"] = None
    elif row["PAUREUS"] < 5:
        row["PAUREUS"] = 0
    else:
        row["PAUREUS"] = 1

    return row

In [ ]:
# Apply the binarization function to each row
df = df.apply(binarize_row, axis=1)

# Aggregate results
df["Result"] = (
    df[["EXACT", "PAUORIG"]]
    .max(axis=1)
    .combine_first(df["TEXT"])
    .combine_first(df["VALUE"])
    .combine_first(df["PAUREUS"])
)

In [ ]:
df.columns = [x + "_binary" for x in df.columns]

In [ ]:
df = df[
    [
        "PAUORIG_binary",
        "EXACT_binary",
        "TEXT_binary",
        "PAUREUS_binary",
        "VALUE_binary",
        "Result_binary",
    ]
]
df.rename(columns={"VALUE_binary": "percentage_inh_binary"}, inplace=True)

In [ ]:
df.shape

(2194772, 6)

In [ ]:
data.shape

(2194772, 40)

In [ ]:
data = data.loc[:, ~data.columns.str.contains("_binary")]

In [ ]:
data = pd.concat([data, df], axis=1)

In [ ]:
data = data[data["INCHI"] != ""]

In [ ]:
data.shape

(2194082, 41)

In [ ]:
data = data[data["Result_binary"].notnull()]

# Annotate lineage

I manually checked all the species names, extended the ones that were abreviated and corrected the ones that contained extra info apart from species

In [ ]:
data = data[data['std_species_final']!='-']

In [ ]:
data.shape

(1747494, 43)

In [ ]:
from Bio import Entrez

# Set your email (NCBI requires this for API access)
Entrez.email = "your_email"

def get_ncbi_taxonomy(species_name):
    """Fetches NCBI Taxonomy lineage for a given species."""
    # Step 1: Search for the Taxonomy ID
    search_handle = Entrez.esearch(db="taxonomy", term=species_name)
    search_results = Entrez.read(search_handle)
    search_handle.close()
    
    if not search_results["IdList"]:
        return f"No results found for {species_name}"
    
    tax_id = search_results["IdList"][0]  # Get first matching TaxID

    # Step 2: Retrieve taxonomic details
    fetch_handle = Entrez.efetch(db="taxonomy", id=tax_id, retmode="xml")
    fetch_results = Entrez.read(fetch_handle)
    fetch_handle.close()

    tax_data = fetch_results[0]
    lineage = tax_data["Lineage"]  # Get full lineage
    
    return {"species": species_name, "tax_id": tax_id, "lineage": lineage}

# Example: Get lineage for Escherichia coli
species = "Fusarium incarnatum-equiseti"
taxonomy_info = get_ncbi_taxonomy(species)
print(taxonomy_info)

No results found for Fusarium incarnatum-equiseti


In [ ]:
import time
from tqdm import tqdm
failed = []
lineage, lineage_full = {},{}
for species in tqdm(data['species_names_complete'].unique()):
    try:
        taxonomy_info = get_ncbi_taxonomy(species)
        time.sleep(1)
        if ' Bacteria;' in taxonomy_info['lineage']:
            lineage[species] = 'Bacteria'
        elif ' Fungi;' in taxonomy_info['lineage']:
            lineage[species] = 'Fungi'
        else:
            lineage[species] = 'Other'
        lineage_full[species] = taxonomy_info['lineage']
    except:
        failed.append(species)

100%|██████████| 1772/1772 [54:53<00:00,  1.86s/it] 


In [ ]:
failed2 = []
for s in tqdm(failed):
    try:
        taxonomy_info = get_ncbi_taxonomy(s)
        time.sleep(1)
        if ' Bacteria;' in taxonomy_info['lineage']:
            lineage[s] = 'Bacteria'
        elif ' Fungi;' in taxonomy_info['lineage']:
            lineage[s] = 'Fungi'
        else:
            lineage[s] = 'Other'
        lineage_full[s] = taxonomy_info['lineage']
    except:
        failed2.append(s)

100%|██████████| 237/237 [05:46<00:00,  1.46s/it]


In [ ]:
get_ncbi_taxonomy('bacillus oleronius')

{'species': 'bacillus oleronius',
 'tax_id': '38875',
 'lineage': 'cellular organisms; Bacteria; Bacillati; Bacillota; Bacilli; Bacillales; Bacillaceae; Heyndrickxia'}

In [ ]:
len(failed2)

216

In [ ]:
list(np.array(failed2)[np.isin(failed2, list(data[data['Result_binary']==1]['species_names_complete'].unique()))])

[np.str_('Candida duobushaemulonis'),
 np.str_('Phomopsis adianticola'),
 np.str_('Fusarium incarnatum-equiseti clinical')]

In [ ]:
data['lineage'] = data['species_names_complete'].map(lineage)

In [ ]:
data[data['Result_binary']==1][['species_names_complete','lineage']].drop_duplicates()['lineage'].value_counts()

lineage
Bacteria    862
Fungi       402
Other        29
Name: count, dtype: int64

In [ ]:
data[np.logical_and(data['Result_binary']==1,data['lineage']=='Other')]['species_names_complete'].unique()

array(['Caenorhabditis elegans', 'Phytophthora pini',
       'Phytophthora capsici', 'G. verrucosa', 'Entamoeba histolytica',
       'Phytophthora cinnamomi', 'Phytophthora melonis', 'Vicia faba',
       'Potentilla indica', 'Prorocentrum donghaiense',
       'Phytophthora nicotianae', 'Leishmania mexicana',
       'Naegleria fowleri', 'Chlamys albidus', 'Phaseolus vulgaris',
       'Halobacterium salinarum', 'Salilagenidium thermophilum',
       'Sclerospora graminicola', 'Saprolegnia parasitica',
       'Pteronura brasiliensis', 'Echinococcus granulosus',
       'Acanthamoeba castellanii', 'Trichomonas vaginalis',
       'Globisporangium spinosum', 'Cylindrotheca closterium',
       'Ulva intestinalis', 'Nicotiana tabacum', 'Hordeum vulgare',
       'Chattonella marina'], dtype=object)

In [ ]:
data[data['Result_binary']==1]['species_names_complete'].nunique()

1296

In [ ]:
positives = data[data['Result_binary']==1].copy()

In [ ]:
positives.columns

Index(['XRN', 'INCHI', 'STR', 'mol', 'smiles', 'TARGET', 'VTYPE', 'PAUREUS',
       'BIOID', 'VALUE', 'UNIT', 'UNIPROT', 'SPECIE', 'AFTYPE', 'BTISSUE',
       'EFFECT', 'CATEG', 'PAUORIG', 'EXACT', 'TEXT', 'BNAME', 'BCELL',
       'BTISSUE.1', 'MODEL', 'PATHO', 'STRAIN', 'ADESC', 'indexes',
       'std_smiles', 'std_species', 'std_species_alt', 'is_same',
       'std_species_final', 'std_species_abbreviated', 'new PAUREUS',
       'PAUORIG_binary', 'EXACT_binary', 'TEXT_binary', 'PAUREUS_binary',
       'percentage_inh_binary', 'Result_binary', 'species_names_complete',
       'lineage'],
      dtype='object')

In [ ]:
# Create a binary matrix: unique compounds as rows, species as columns
bacteria_binary_matrix = positives[positives['lineage']=='Bacteria'][['species_names_complete','INCHI']].drop_duplicates().pivot_table(index="INCHI", columns="species_names_complete", 
                                           aggfunc=lambda x: 1, fill_value=0)

# Display the first few rows
print(bacteria_binary_matrix.shape)


(97431, 862)


# NCBI species names

The following code was used to obtain the NCBI names from the original Reaxys annotated names.

In [ ]:
# --- helpers ---
def _esearch(term, retmax=20, retries=3):
    for attempt in range(retries):
        try:
            with Entrez.esearch(db="taxonomy", term=term, retmode="xml", retmax=retmax) as h:
                rec = Entrez.read(h)
            return rec.get("IdList", [])
        except HTTPError as e:
            if e.code in (429, 500, 502, 503, 504) and attempt < retries - 1:
                sleep(0.8 * (attempt + 1))
                continue
            raise
    return []

def _esummary(ids):
    if not ids:
        return []
    with Entrez.esummary(db="taxonomy", id=",".join(ids), retmode="xml") as h:
        recs = Entrez.read(h)
    # Normalize a tiny bit
    out = []
    for r in recs:
        out.append({
            "taxid": str(r.get("Id")),
            "name": r.get("ScientificName"),
            "rank": r.get("Rank"),
            "division": r.get("Division"),
            "lineage": r.get("Lineage") or "",
        })
    return out

def _sanitize_name(name):
    # Remove obvious strain-like tails to improve matches, e.g. "Escherichia coli K-12 MG1655"
    parts = name.replace("_", " ").split()
    if len(parts) >= 2:
        # keep binomial, drop extras
        return " ".join(parts[:2])
    return name

def _best_bacterial_hit(summaries, prefer_ranks=("species","subspecies","no rank")):
    # keep only bacterial lineage
    bact = [s for s in summaries if "Bacteria" in s["lineage"] or s["division"] == "Bacteria"]
    if not bact:
        return None
    # prefer desired ranks in order
    for rk in prefer_ranks:
        cand = [s for s in bact if (s["rank"] or "").lower() == rk]
        if cand:
            # if multiple, pick the one whose name appears simplest/closest (shortest name)
            cand.sort(key=lambda x: (len(x["name"] or ""), int(x["taxid"])))
            return cand[0]
    # otherwise smallest taxid as a stable fallback
    bact.sort(key=lambda x: int(x["taxid"]))
    return bact[0]

def name_to_taxid(name, allow_fallback=True):
    """
    Returns a dict with taxid, matched_name, rank, lineage — or None if not found.
    """
    raw = name.strip()
    clean = _sanitize_name(raw)

    # 1) Try exact scientific name within Bacteria
    term1 = f'("{raw}"[Scientific Name] OR "{clean}"[Scientific Name]) AND Bacteria[Superkingdom]'
    ids = _esearch(term1, retmax=25)
    summaries = _esummary(ids)
    best = _best_bacterial_hit(summaries)
    if best:
        return best

    # 2) Try All Names (catches synonyms, typos that NCBI knows)
    term2 = f'("{raw}"[All Names] OR "{clean}"[All Names]) AND Bacteria[Superkingdom]'
    ids = _esearch(term2, retmax=25)
    summaries = _esummary(ids)
    best = _best_bacterial_hit(summaries)
    if best:
        return best

    # 3) Optional: broader fallback without forcing Bacteria (rare but helps edge cases)
    if allow_fallback:
        term3 = f'("{raw}"[All Names] OR "{clean}"[All Names])'
        ids = _esearch(term3, retmax=25)
        summaries = _esummary(ids)
        best = _best_bacterial_hit(summaries) or (summaries[0] if summaries else None)
        if best:
            return best

    return None

# --- batch mapping with courteous pacing (<=3 req/s without API key) ---
def map_names_to_taxids(names):
    out = {}
    for n in names:
        hit = name_to_taxid(n)
        out[n] = hit["taxid"] if hit else None
        # Be polite to NCBI: ~0.34s keeps us under 3 requests/second without an API key
        sleep(0.34 if Entrez.api_key is None else 0.12)
    return out

def map_names_to_NCBI_names(names):
    out = {}
    for n in names:
        hit = name_to_taxid(n)
        out[n] = hit["name"] if hit else None
        # Be polite to NCBI: ~0.34s keeps us under 3 requests/second without an API key
        sleep(0.34 if Entrez.api_key is None else 0.12)
    return out
# --- example ---
# names = ["Escherichia coli", "Bacillus subtilis", "Pseudomonas aeruginosa"]
# mapping = map_names_to_taxids(names)
# print(mapping)

In [ ]:
new_names = map_names_to_NCBI_names(species)